## Reading from bronze

In [0]:
df = spark.read.table("workspace.bronze.crm_cust_info")
display(df)

## Data Transformations

In [0]:
# imports
from pyspark.sql.types import StringType 
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col

In [0]:
#trim

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


In [0]:
#normalisation
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

In [0]:
# handle missing values
df = df.filter(col("cst_id").isNotNull())

In [0]:
# renaming friendly
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(10).display()

##Write into silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

Sanity check

In [0]:

%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 10